In [ ]:
import sys
import os
import nest_asyncio

project_root = os.path.abspath('.')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

nest_asyncio.apply()

print(f'Project root set to: {project_root}')

In [ ]:
from app.db.connection import SessionLocal, engine, Base
from app.db.repository import Repository

Base.metadata.create_all(bind=engine)

db = SessionLocal()
try:
    conv = Repository.create_conversation(db, user_id='test_user_notebook', entry_point='notebook_test')
    conv_id = str(conv.id)
    print(f'Created Conversation ID: {conv_id}')
    msg = Repository.save_message(db, conv_id, role='user', content='Hello from notebook!')
    print(f'Saved Message ID: {str(msg.id)}')
finally:
    db.close()

In [ ]:
from app.db.connection import SessionLocal
from app.guardrails.input_guardrail import InputGuardrail
from app.services.recommendation_service import RecommendationService
from app.retrieval.hybrid_retriever import HybridRetriever
from app.services.context_service import ContextService
from app.services.assistant_service import AssistantService

db = SessionLocal()
try:
    user_input = 'What snacks support my school?'
    clean_input = InputGuardrail.validate(user_input)
    print(f'[1] Guardrail Output: {clean_input}')
    intent = RecommendationService.classify_intent(clean_input)
    print(f'[2] Classified Intent: {intent}')
    facts = HybridRetriever.retrieve(db, intent, clean_input)
    print(f'[3] SQL Retrieved Facts: {facts}')
    context = ContextService.prepare_context(intent, facts)
    print(f'[4] Prepared Prompt Context:\n{context}')
    msg_db, intent, reply, facts = AssistantService.process_message(db, conv_id, clean_input)
    print(f'[5] Final Assistant Reply:\n{reply}')
finally:
    db.close()

In [ ]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

res = client.get('/')
print('Health Check:', res.json())

create_res = client.post('/v1/chat/conversations', json={'entry_point': 'notebook_test'})
conv_data = create_res.json()
print('Create Conversation Response:', conv_data)

conv_id = conv_data['conversation_id']

msg_res = client.post(
    f'/v1/chat/conversations/{conv_id}/messages',
    json={'content': "Can I buy Annie's Macaroni?"}
)
print('Send Message Response:', msg_res.json())

In [3]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

# Start a new conversation
create_res = client.post('/v1/chat/conversations', json={'entry_point': 'interactive'})
conv_id = create_res.json()['conversation_id']
print(f'Conversation started: {conv_id}')
print('Type your message below and run this cell. Change the prompt and re-run for multi-turn.\n')

# --- Change this prompt and re-run the cell ---
user_prompt = "How do I refer a friend?"



# ----------------------------------------------

res = client.post(
    f'/v1/chat/conversations/{conv_id}/messages',
    json={'content': user_prompt}
)
data = res.json()
print(f'You: {user_prompt}')
print(f'Capper [{data["intent"]}]: {data["reply"]}')

Conversation started: c_0f8a4fd4
Type your message below and run this cell. Change the prompt and re-run for multi-turn.

You: How do I refer a friend?
Capper [GENERAL]: Referring a friend can vary depending on the context (e.g., a job, a service, or a product). Here are some general steps you can follow:

1. **Choose the Right Method**: Decide how you want to make the referral. This could be through an online form, email, social media, or in-person communication.

2. **Provide Context**: When making the referral, briefly explain why you think your friend would be a good fit or why you’re recommending them. Mention any relevant skills or experiences.

3. **Include Necessary Information**: If applicable, provide your friend's contact information, resume, or any other required details.

4. **Personal Touch**: If you’re communicating directly, add a personal note about your friend to make it more genuine.

5. **Follow Up**: If you’re referring someone for a job, it’s a good idea to let yo